# Evaluation
Compares three configurations on `eval/questions.json` (15 answerable + 4 unanswerable questions):

| Config | Retrieval | Chunk size |
|---|---|---|
| no_rag | — | — |
| rag_200 | top-5 | ~200 tokens |
| rag_400 | top-5 | ~400 tokens |

**Metrics**
- **Retrieval:** hit@5 — the expected article is among the top-5 chunks (no LLM involved).
- **Answers (LLM judges):** correctness vs. the expected answer, relevance to the question, groundedness in the retrieved context (hallucination risk, RAG only).
- **Missing context:** on unanswerable questions the assistant must say it does not know; on answerable ones it must not refuse.
- **Cost:** prompt tokens and latency per question.

In [0]:
%pip install -q databricks-vectorsearch openai "mlflow>=3"

In [0]:
%restart_python

In [0]:
import json
import os
import sys

sys.path.append(os.path.abspath(".."))

import mlflow
import pandas as pd
from mlflow.genai.scorers import Correctness, RelevanceToQuery, RetrievalGroundedness

from rag.config import CHUNK_SIZES
from rag.qa import NO_ANSWERS, ask, retrieve

In [0]:
USER = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{USER}/lab12-f1-rag")

with open(os.path.abspath("../eval/questions.json"), encoding="utf-8") as f:
    QUESTIONS = json.load(f)

answerable = [q for q in QUESTIONS if q["answerable"]]

### 1. Retrieval quality: hit@5
For every answerable question and every chunk size: is the expected article among the top-5 results, and at which rank. No category filter — this measures plain search.

In [0]:
retrieval_rows = []
for size in CHUNK_SIZES:
    for q in answerable:
        titles = [c["title"] for c in retrieve(q["question"], k=5, chunk_size=size)]
        rank = titles.index(q["expected_title"]) + 1 if q["expected_title"] in titles else None
        retrieval_rows.append({
            "chunk_size": size, 
            "id": q["id"], 
            "question": q["question"],
            "expected_title": q["expected_title"], 
            "hit": rank is not None,
            "rank": rank, 
            "retrieved_titles": titles
        })

retrieval_df = pd.DataFrame(retrieval_rows)
display(
    retrieval_df.groupby("chunk_size")
        .agg(hit_at_5=("hit", "mean"), avg_rank_when_hit=("rank", "mean"))
        .round(2)
        .reset_index()
)

### 2. Answer quality: three configurations
`mlflow.genai.evaluate` runs `ask()` for every question and scores the answers with LLM judges.

`RetrievalGroundedness` reads the context from the retriever span, so it is used only for RAG configs.


In [0]:
CONFIGS = {
    "no_rag":  {"use_retrieval": False, "chunk_size": None},
    **{f"rag_{size}": {"use_retrieval": True, "chunk_size": size} for size in CHUNK_SIZES},
}

eval_data = [
    {"inputs": {"question": q["question"]},
     "expectations": {"expected_response": q["expected_answer"]}}
    for q in answerable
]
unanswerable = [q for q in QUESTIONS if not q["answerable"]]

answers = []      # full ask() results: tokens, latency, refusals
judge_metrics = {}

for name, cfg in CONFIGS.items():
    def predict_fn(question, _cfg=cfg, _name=name):
        kwargs = {"use_retrieval": _cfg["use_retrieval"]}
        if _cfg["chunk_size"]:
            kwargs["chunk_size"] = _cfg["chunk_size"]
        r = ask(question, **kwargs)
        answers.append({"config": _name, **r})
        return r["answer"]

    scorers = [Correctness(), RelevanceToQuery()]
    if cfg["use_retrieval"]:
        scorers.append(RetrievalGroundedness())

    with mlflow.start_run(run_name=f"eval_{name}"):
        result = mlflow.genai.evaluate(data=eval_data, predict_fn=predict_fn, scorers=scorers)
    
    for q in unanswerable:
        predict_fn(q["question"])
    
    judge_metrics[name] = result.metrics

### 3. Missing context, cost and summary
- **honest_refusals** — unanswerable questions answered with "I don't know"
- **false_refusals** — answerable questions the assistant refused

In [0]:
answers_df = pd.DataFrame(answers)
answerable_ids = {q["question"]: q["answerable"] for q in QUESTIONS}
answers_df["answerable"] = answers_df["question"].map(answerable_ids)
answers_df["refused"] = answers_df["answer"].str.startswith(NO_ANSWERS)

summary = []
for name in CONFIGS:
    a = answers_df[answers_df["config"] == name]
    m = judge_metrics[name]
    size = CONFIGS[name]["chunk_size"]
    hits = retrieval_df[retrieval_df["chunk_size"] == size]["hit"].mean() if size else None
    summary.append({
        "config": name,
        "hit_at_5": hits,
        "correctness": m.get("correctness/mean"),
        "relevance": m.get("relevance_to_query/mean"),
        "groundedness": m.get("retrieval_groundedness/mean"),
        "honest_refusals": a[~a["answerable"]]["refused"].mean(),
        "false_refusals": a[a["answerable"]]["refused"].mean(),
        "avg_prompt_tokens": a["prompt_tokens"].mean(),
        "avg_latency_ms": a["latency_ms"].mean(),
    })

summary_df = pd.DataFrame(summary).round(2)
display(summary_df)

### 4. Examples side by side
Recent facts (q02, q05) and an off-topic question (q18) show the difference most clearly.

In [0]:
example_questions = [q["question"] for q in QUESTIONS if q["id"] in ("q02", "q05", "q18")]
examples = (
    answers_df[answers_df["question"].isin(example_questions)]
        .pivot(index="question", columns="config", values="answer")
        .reset_index()
)
display(examples)